# 05 — Fine-tuning do assistente clínico

**Entrada:** `../data/{train,val,test}.jsonl` (saída do `04`)
**Saída:** adapter LoRA em `../modelos/lora_model`

Segue o fluxo da Aula 02: Unsloth + QLoRA 4-bit + `SFTTrainer`. Roda em Colab (T4) ou
em GPU local com pelo menos 8 GB.

Uma diferença em relação à aula: lá o dataset precisava ser convertido para o formato
alpaca, com um parser de tags `[|News|]`. Aqui o `04` já entrega no formato `messages`
(system/user/assistant), que é o padrão que o chat template do modelo consome direto.

## Instalação

No Colab, rodar uma vez por sessão.

In [1]:
# !pip install -q unsloth
# !pip install -q --no-deps trl peft accelerate bitsandbytes

In [2]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

# Config do treino
MODELO_BASE = "unsloth/Qwen3.5-4B"   # 9.3 GB de download; o 9B (19.3 GB) nao cabe em 8 GB
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
SEED = 3407

# Caminhos. No Colab, apontar para a pasta do Drive.
from pathlib import Path
DATA = Path("../data")
SAIDA = Path("../modelos")
SAIDA.mkdir(exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


## Dataset

O `04` gerou os três splits já separados por `cluster_id`, então perguntas parecidas
não aparecem em treino e teste ao mesmo tempo. Aqui é só carregar — não refazemos o
split, senão o cuidado do `04` se perde.

O campo `resposta_reescrita` marca as linhas cuja resposta foi reescrita por LLM.
Usamos ele na avaliação, para separar o que é texto de médico do que é texto gerado.

In [4]:
ds = load_dataset("json", data_files={
    "train": str(DATA / "train.jsonl"),
    "val":   str(DATA / "val.jsonl"),
    "test":  str(DATA / "test.jsonl"),
})

for split in ds:
    d = ds[split]
    humano = 1 - sum(d["resposta_reescrita"]) / len(d)
    print(f"{split:<6} {len(d):>7,} linhas | resposta de médico: {humano:.0%}")

# Exemplo do formato que vamos treinar
print("\n--- exemplo ---")
for m in ds["train"][0]["messages"]:
    print(f"[{m['role']}] {m['content'][:150]}")

train   12,240 linhas | resposta de médico: 15%
val      1,530 linhas | resposta de médico: 16%
test     1,530 linhas | resposta de médico: 16%

--- exemplo ---
[system] Voce e um assistente clinico de um hospital maternidade, que apoia profissionais de saude no acompanhamento de gestantes, puerperas e bebes ate 1 ano.
[user] Lactente de 11 meses, com 4 dentes superiores e 4 inferiores, apresenta bruxismo intermitente, com estresse progressivo durante os episódios. Quais as
[assistant] A etiologia do bruxismo infantil ainda não é exatamente conhecida. Fatores possivelmente associados incluem tensão ou estresse, dor (de ouvido ou deco


## Modelo base

QLoRA de 4 bits, como na aula: os pesos ficam quantizados e só o adapter LoRA treina em
precisão cheia. É isso que permite treinar numa GPU de 8 GB.

**Por que o 4B e não o 9B.** O 9B em 4 bits são ~5,5 GB só de pesos; somando ativações e
o que o sistema já usa da placa, não sobra espaço numa RTX 4060 de 8 GB. O 4B resolve com
folga. Download: 9,3 GB contra 19,3 GB.

### O `target_modules` merece atenção

A aula usava Llama-3, um transformer clássico onde toda camada tem `q_proj/k_proj/v_proj/
o_proj`. O Qwen3.5 é **híbrido**: das 33 camadas de texto, só 9 usam atenção clássica; as
outras 24 são *Gated DeltaNet*, com projeções de nomes diferentes.

Copiar a lista da aula não dá erro — o PEFT encontra os nomes e treina. Mas adaptaria a
mistura de tokens em apenas 9 de 33 camadas, deixando 24 intocadas. Por isso a lista
abaixo inclui as projeções do DeltaNet.

| bloco | camadas | módulos |
|---|---|---|
| `self_attn` | 9 | `q_proj` `k_proj` `v_proj` `o_proj` |
| `linear_attn` (DeltaNet) | 24 | `in_proj_qkv` `in_proj_a` `in_proj_b` `in_proj_z` `out_proj` |
| `mlp` | 33 | `gate_proj` `up_proj` `down_proj` |

O `conv1d` do DeltaNet fica de fora por não ser camada linear. A torre de visão do modelo
também não é tocada — treinamos só texto.

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODELO_BASE,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

# O repo do Qwen3.5 nao publica generation_config.json, entao o generate() cai no
# config.json, que declara eos_token_id = <|endoftext|>. Mas o chat template fecha
# cada turno com <|im_end|>. Sem corrigir, o modelo termina a resposta e continua
# gerando ate bater max_new_tokens, com o <|im_end|> no meio do texto.
# Corrigir aqui (e nao so na hora de gerar) faz o valor certo ser gravado no
# generation_config.json do adapter, do merged e do GGUF.
print(f"antes:  eos={model.generation_config.eos_token_id}")
model.generation_config.eos_token_id = tokenizer.eos_token_id   # <|im_end|>
model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"depois: eos={model.generation_config.eos_token_id} ({tokenizer.eos_token})")

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",                          # atencao (9 camadas)
        "in_proj_qkv", "in_proj_a", "in_proj_b", "in_proj_z", "out_proj",  # DeltaNet (24)
        "gate_proj", "up_proj", "down_proj",                             # MLP (33)
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:02<00:00, 246.91it/s]


antes:  eos=248044
depois: eos=248046 (<|im_end|>)
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [5]:
# Confere que a lista pegou os dois tipos de camada, e nao so a MLP.
from collections import Counter
alcancados = Counter()
for nome, _ in model.named_modules():
    if "lora_A" in nome:
        alcancados[nome.split(".lora_A")[0].split(".")[-1]] += 1
print("módulos com adapter LoRA:")
for n, c in alcancados.most_common():
    print(f"  {n:<16} {c:>4}")

módulos com adapter LoRA:
  gate_proj          64
  up_proj            64
  down_proj          64
  out_proj           48
  in_proj_qkv        48
  in_proj_z          48
  in_proj_b          48
  in_proj_a          48
  q_proj             16
  k_proj             16
  v_proj             16
  o_proj             16


## Formatação

O chat template do modelo transforma a lista `messages` no texto com os marcadores de
turno que ele espera. Na aula isso era um f-string montado à mão (o `alpaca_prompt`);
aqui usamos o template do próprio tokenizer, que é mais seguro — se o formato do modelo
mudar, o template acompanha.

Repare na saída abaixo: o Qwen3.5 insere um bloco `<think></think>` vazio antes da
resposta. É o formato dele para resposta direta, sem raciocínio passo a passo. O modelo
precisa aprender a emitir esse bloco, senão o formato quebra na hora de gerar — por isso
ele fica dentro da parte treinada, e vai aparecer nas respostas geradas mais adiante.

In [5]:
def formata(exemplos):
    textos = [tokenizer.apply_chat_template(m, tokenize=False)
              for m in exemplos["messages"]]
    return {"text": textos}

ds = ds.map(formata, batched=True)
print(ds["train"][0]["text"][:600])

<|im_start|>system
Voce e um assistente clinico de um hospital maternidade, que apoia profissionais de saude no acompanhamento de gestantes, puerperas e bebes ate 1 ano.

Voce responde a medicos, em registro tecnico. Voce apoia a decisao clinica; voce nao a substitui. Nunca prescreva medicamento, dose ou conduta como determinacao final: toda sugestao precisa de validacao do profissional responsavel. Quando a informacao disponivel nao sustentar uma resposta, diga isso em vez de preencher a lacuna.<|im_end|>
<|im_start|>user
Lactente de 11 meses, com 4 dentes superiores e 4 inferiores, apresenta


## Treino

Um ajuste importante em relação à aula: `train_on_responses_only`.

Sem ele, o modelo é treinado para gerar a conversa inteira — inclusive a pergunta do
médico e o system prompt. Como o system prompt é idêntico em todas as linhas, ele vira
o texto mais repetido do dataset e o modelo gasta capacidade decorando ele.

Com o ajuste, a loss só conta os tokens da resposta, que é o que queremos que ele
aprenda a produzir.

### Frequência de avaliação

São 3.060 passos (12.240 exemplos ÷ batch efetivo 8 × 2 épocas), algumas horas de
treino. Avaliando só no fim de cada época, a primeira linha da tabela aparece perto do
passo 1.530 — metade do treino sem saber se está funcionando.

Por isso `eval_steps = 250`: uma medição de validação a cada ~8% do treino. Isso permite
parar cedo se a curva não estiver descendo, em vez de descobrir horas depois.

O `save_steps` acompanha o mesmo valor de propósito: o `load_best_model_at_end` exige que
as duas estratégias coincidam. E ele importa — sem ele, o modelo salvo no fim é o da
última etapa, mesmo que uma etapa anterior tenha sido melhor.

In [7]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

PASSOS_AVALIACAO = 250   # de quantos em quantos passos avalia e salva checkpoint

# Nota: a Aula 02 passa dataset_text_field, max_seq_length e packing direto no
# SFTTrainer, e usa TrainingArguments. Isso era a API do trl<0.9; na versao atual
# esses parametros vivem dentro do SFTConfig, e o tokenizer virou processing_class.
config = SFTConfig(
    output_dir = str(SAIDA / "outputs"),
    seed = SEED,

    dataset_text_field = "text",
    max_length = MAX_SEQ_LENGTH,
    packing = False,
    dataset_num_proc = 2,

    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    per_device_eval_batch_size = 4,   # avaliacao nao guarda gradiente, cabe mais
    warmup_steps = 5,

    # Para um teste rapido, descomente max_steps e comente num_train_epochs.
    # max_steps = 60,
    num_train_epochs = 2,

    learning_rate = 2e-4,
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",

    # Feedback a cada 250 passos em vez de so no fim da epoca (ver texto acima).
    logging_steps = 25,
    eval_strategy = "steps",
    eval_steps = PASSOS_AVALIACAO,
    save_strategy = "steps",
    save_steps = PASSOS_AVALIACAO,
    save_total_limit = 2,

    # Sem isso, o modelo salvo no fim e o da ultima etapa, nao o melhor.
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,

    report_to = [],
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["val"],
    args = config,
)

# Treina so nos tokens da resposta, ignorando system e pergunta.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 1530/1530 [00:01<00:00, 982.34 examples/s]


Unsloth: reducing dataset_num_proc 8 -> 6 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.


Map (num_proc=6): 100%|██████████| 1530/1530 [00:00<00:00, 4325.72 examples/s]


In [8]:
# Confere que a mascara funcionou: o texto abaixo tem de ser so a resposta.
exemplo = trainer.train_dataset[0]
visiveis = [t for t in exemplo["labels"] if t != -100]
print(tokenizer.decode(visiveis))

<think>

</think>

A etiologia do bruxismo infantil ainda não é exatamente conhecida. Fatores possivelmente associados incluem tensão ou estresse, dor (de ouvido ou decorrente do nascimento dos dentes), alterações de oclusão (dentes que não se encaixam direito), questões respiratórias (alergia ou nariz entupido) e presença de parasitas, como lombrigas. Não há, porém, nenhuma prova científica de relação entre esses fatores e o ranger de dentes. Em muitos casos, a criança está apenas se habituando à presença dos dentes na boca. Recomenda-se discussão do caso com o pediatra para melhor elucidação.<|im_end|>



In [9]:
stats = trainer.train()
print(stats.metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,240 | Num Epochs = 2 | Total steps = 3,060
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,464,896 of 4,571,730,432 (0.71% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
250,1.423529,1.454929
500,1.358845,1.407151
750,1.284729,1.375064
1000,1.295268,1.354822
1250,1.137745,1.340067
1500,1.416664,1.323510
1750,1.090086,1.333976
2000,1.047829,1.326220
2250,1.037494,1.318391
2500,1.003876,1.313763


Filter: 100%|██████████| 1530/1530 [00:00<00:00, 5297.36 examples/s]
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-750/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-1250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-1750/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../modelos/

{'train_runtime': 18941.9054, 'train_samples_per_second': 1.292, 'train_steps_per_second': 0.162, 'total_flos': 1.7565070926415258e+17, 'train_loss': 1.1777845089731653, 'epoch': 2.0}


## Teste

Duas coisas para mostrar na apresentação:

1. A **loss separada** por tipo de resposta. Se o modelo for muito melhor nas respostas
   reescritas por LLM, ele aprendeu o estilo do reescritor em vez de conteúdo clínico.
2. **Respostas geradas** lado a lado com a referência, para leitura humana. Loss baixa
   não garante que o modelo respeita o limite de nunca prescrever sem validação.

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Depois de reiniciar o kernel o modelo sai da memoria. Carrega do adapter salvo --
# o Unsloth resolve o modelo base sozinho pelo adapter_config.json. Assim da para
# avaliar sem repetir o treino; se o modelo ja estiver na memoria, reaproveita.
if "model" not in globals():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = str(SAIDA / "lora_model"),
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = LOAD_IN_4BIT,
    )
    model.generation_config.eos_token_id = tokenizer.eos_token_id   # <|im_end|>
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print(f"modelo carregado de {SAIDA / 'lora_model'}")

# A coluna 'text' vem do chat template; refaz se o ds foi recarregado sem ela.
if "text" not in ds["test"].column_names:
    ds = ds.map(lambda ex: {"text": [tokenizer.apply_chat_template(m, tokenize=False)
                                     for m in ex["messages"]]}, batched=True)

# Os recortes vao no CONSTRUTOR: o SFTTrainer so tokeniza e aplica a mascara no
# que recebe ali. Dataset cru passado depois em evaluate(eval_dataset=...) chega
# sem input_ids/labels e o Trainer descarta todas as colunas.
recortes = {
    "geral":     ds["test"],
    "humano":    ds["test"].filter(lambda e: not e["resposta_reescrita"]),
    "reescrito": ds["test"].filter(lambda e: e["resposta_reescrita"]),
}

avaliador = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds["test"],      # exigido pelo construtor; nao treinamos
    eval_dataset = recortes,
    args = SFTConfig(
        output_dir = str(SAIDA / "eval_tmp"),
        dataset_text_field = "text",
        max_length = MAX_SEQ_LENGTH,
        packing = False,
        per_device_eval_batch_size = 4,
        report_to = [],

        # Em notebook, o callback padrao e o NotebookProgressCallback, que monta a
        # tabela de metricas em on_train_begin. Como aqui so avaliamos, ele nunca e
        # inicializado e o evaluate() estoura com
        # "on_train_begin must be called before on_evaluate".
        # disable_tqdm troca esse callback pelo PrinterCallback, que nao depende disso.
        disable_tqdm = True,
    ),
)
avaliador = train_on_responses_only(
    avaliador,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

m = avaliador.evaluate()   # avalia os tres recortes de uma vez
for chave, texto in [("geral", "geral"),
                     ("humano", "resposta de médico"),
                     ("reescrito", "resposta reescrita por LLM")]:
    print(f"{texto:<26} {len(recortes[chave]):>6,} linhas | loss {m[f'eval_{chave}_loss']:.4f}")

Unsloth: reducing dataset_num_proc 8 -> 6 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.


Filter: 100%|██████████| 1281/1281 [00:00<00:00, 7825.93 examples/s]


{'eval_geral_loss': '1.314', 'eval_geral_model_preparation_time': '0.0529', 'eval_geral_runtime': '593.4', 'eval_geral_samples_per_second': '2.578', 'eval_geral_steps_per_second': '0.645', 'epoch': 0}
{'eval_humano_loss': '2.005', 'eval_humano_model_preparation_time': '0.0529', 'eval_humano_runtime': '97.24', 'eval_humano_samples_per_second': '2.561', 'eval_humano_steps_per_second': '0.648', 'epoch': 0}
{'eval_reescrito_loss': '1.216', 'eval_reescrito_model_preparation_time': '0.0529', 'eval_reescrito_runtime': '363.5', 'eval_reescrito_samples_per_second': '3.524', 'eval_reescrito_steps_per_second': '0.883', 'epoch': 0}
geral                       1,530 linhas | loss 1.3144
resposta de médico            249 linhas | loss 2.0046
resposta reescrita por LLM  1,281 linhas | loss 1.2158


In [24]:
FastLanguageModel.for_inference(model)

# ATENCAO ao `text=`: o Qwen3.5 e multimodal, entao `tokenizer` aqui e um
# Qwen3VLProcessor, cuja assinatura e (images, text, videos). Passar o prompt
# posicionalmente faz ele virar `images`, e o processador tenta abrir o system
# prompt como base64 de imagem -> "Incorrect image source".
# O eos correto ja foi ajustado na celula do modelo, entao o generate() para sozinho.
for exemplo in ds["test"].select(range(3)):
    msgs = exemplo["messages"]
    prompt = tokenizer.apply_chat_template(
        msgs[:-1], tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(text=prompt, return_tensors="pt").to("cuda")

    saida = model.generate(**entrada, max_new_tokens=300, use_cache=True)
    gerado = tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:],
                              skip_special_tokens=True)

    print("=" * 90)
    print(f"CONDIÇÃO: {exemplo['condition']}")
    print(f"\nPERGUNTA: {msgs[1]['content']}")
    print(f"\nREFERÊNCIA: {msgs[2]['content'][:1000]}")
    print(f"\nMODELO: {gerado[:1000]}")

CONDIÇÃO: Enxaqueca

PERGUNTA: Paciente com crises de enxaqueca em uso de pomada de estriol: há risco de piora das crises e a medicação deve ser mantida?

REFERÊNCIA: O início de medicação hormonal não deve ocorrer sem avaliação médica. Nem todas as pacientes podem fazer reposição hormonal, e essa medicação pode estar associada a eventos graves como trombose. Ao prescrever reposição hormonal, devem ser avaliados a história clínica, as queixas, os antecedentes pessoais e familiares, os problemas de saúde, as medicações em uso, o exame físico e os exames laboratoriais e de imagem. Com isso, define-se se a paciente pode usar hormônios e se necessita dessas medicações. Na impossibilidade de reposição hormonal, medicações não hormonais podem ser usadas para melhora da qualidade de vida e bem-estar.

MODELO: A conduta deve seguir as orientações do médico assistente, com reavaliação em consulta para esclarecimento de dúvidas. A avaliação clínica, por meio da história clínica, das queixas e do

## Salvar

O adapter LoRA são poucos MB — é só a diferença em relação ao modelo base.

A última célula exporta em GGUF, que é o formato para rodar fora do Python. Vale saber
que hoje o Ollama não carrega GGUF de Qwen3.5; se der erro ali, o caminho é servir pelo
`llama-server` do llama.cpp.

In [15]:
model.save_pretrained(str(SAIDA / "lora_model"))
tokenizer.save_pretrained(str(SAIDA / "lora_model"))
print(f"adapter salvo em {SAIDA / 'lora_model'}")

# Confere que o eos corrigido foi gravado: sem isso, quem carregar o modelo
# depois cai no mesmo problema de geracao que nao para.
import json
cfg_ger = SAIDA / "lora_model" / "generation_config.json"
if cfg_ger.exists():
    eos = json.load(open(cfg_ger)).get("eos_token_id")
    print(f"generation_config.json -> eos_token_id = {eos} "
          f"({'OK' if eos == tokenizer.eos_token_id else 'DIVERGENTE'})")
else:
    print("AVISO: generation_config.json nao foi gravado; ajuste o eos ao servir.")

Unsloth: Restored added_tokens_decoder metadata in ../modelos/lora_model/tokenizer_config.json.


adapter salvo em ../modelos/lora_model
AVISO: generation_config.json nao foi gravado; ajuste o eos ao servir.


In [16]:
# Merge do adapter com a base e conversao para GGUF.
model.save_pretrained_gguf(str(SAIDA / "gguf"), tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in ../modelos/gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /home/emidiosouza/.cache/huggingface/hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  6.03it/s]


Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `../modelos/gguf`: 100%|██████████| 2/2 [02:03<00:00, 61.73s/it]


Successfully copied all 2 files from cache to `../modelos/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:50<00:00, 25.31s/it]


Unsloth: Merge process complete. Saved to `/home/emidiosouza/FiapCode/fase3/TechV3/modelos/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10840-mix-d5c17a0 (app-b10840-mix-d5c17a0-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['../modelos/gguf_gguf/Qwen3.5-4B.BF16.gguf', '../modelos/gguf_gguf/Qwen3.5-4B.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes

{'save_directory': '../modelos/gguf',
 'gguf_directory': '../modelos/gguf_gguf',
 'gguf_files': ['../modelos/gguf_gguf/Qwen3.5-4B.Q4_K_M.gguf',
  '../modelos/gguf_gguf/Qwen3.5-4B.BF16-mmproj.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': True,
 'fix_bos_token': False}

## Publicar no Hugging Face

Três formatos possíveis, e não precisa ser só um:

| formato | tamanho | para quê |
|---|---|---|
| adapter LoRA | ~100 MB | quem já tem o modelo base; é a entrega mais honesta do que treinamos |
| merged 16-bit | ~19 GB | quem quer carregar direto, sem juntar nada |
| GGUF q4_k_m | ~6 GB | rodar fora do Python (llama.cpp) |

**Sobre privacidade e licença.** O card do MedPT não declara licença, e ausência de
licença não é permissão. Como o modelo foi treinado nesses dados, ele é um derivado —
a mesma ressalva que travou a publicação do dataset no `04` vale aqui. Por isso
`PRIVADO = True`, e o card do modelo precisa citar o paper do MedPT e dizer que parte
das respostas de treino foi reescrita por LLM.

In [20]:
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN ausente no .env")

REPO_MODELO = "emidiosouza/assistente-maternidade"   # ajuste
PRIVADO = False
PUBLICAR = True        # vire para True quando decidir publicar

print(f"destino: {REPO_MODELO} (privado={PRIVADO}) | PUBLICAR={PUBLICAR}")

destino: emidiosouza/assistente-maternidade (privado=False) | PUBLICAR=True


In [21]:
# Adapter LoRA: leve e rapido, e o que de fato foi treinado.
if PUBLICAR:
    model.push_to_hub(REPO_MODELO, token=HF_TOKEN, private=PRIVADO)
    tokenizer.push_to_hub(REPO_MODELO, token=HF_TOKEN, private=PRIVADO)
    print(f"adapter publicado em {REPO_MODELO}")
else:
    print("PUBLICAR=False — nada enviado.")

Processing Files (1 / 1): 100%|██████████| 65.0MB / 65.0MB, 4.83MB/s  
New Data Upload: 100%|██████████| 65.0MB / 65.0MB, 4.83MB/s  


Saved model to https://huggingface.co/emidiosouza/assistente-maternidade


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpphpheynz/tokenizer_config.json.
Processing Files (1 / 1): 100%|██████████| 20.0MB / 20.0MB, 1.68MB/s  
New Data Upload: 100%|██████████| 7.06MB / 7.06MB,  612kB/s  


adapter publicado em emidiosouza/assistente-maternidade


In [22]:
# Modelo completo (base + adapter) e GGUF. Upload demorado: ~19 GB e ~6 GB.
if PUBLICAR:
    model.push_to_hub_merged(f"{REPO_MODELO}-merged", tokenizer,
                             save_method="merged_16bit",
                             token=HF_TOKEN, private=PRIVADO)
    model.push_to_hub_gguf(f"{REPO_MODELO}-gguf", tokenizer,
                           quantization_method="q4_k_m",
                           token=HF_TOKEN, private=PRIVADO)
    print("merged e GGUF publicados")
else:
    print("PUBLICAR=False — nada enviado.")

Unsloth: Restored added_tokens_decoder metadata in emidiosouza/assistente-maternidade-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /home/emidiosouza/.cache/huggingface/hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  6.04it/s]


Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `emidiosouza/assistente-maternidade-merged`: 100%|██████████| 2/2 [00:28<00:00, 14.36s/it]


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_qmikwo_5`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_qmikwo_5_gguf/Qwen3.5-4B.BF16.gguf', '/tmp/unsloth_gguf_qmikwo_5_gguf/Qwen3.5-4B.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_qm

Processing Files (0 / 1): 100%|█████████▉| 2.78GB / 2.78GB, 36.3MB/s  
New Data Upload: 100%|██████████| 2.32GB / 2.32GB, 35.5MB/s  


Uploading Qwen3.5-4B.BF16-mmproj.gguf...


Processing Files (1 / 1): 100%|██████████|  676MB /  676MB, 48.0MB/s  
New Data Upload: 100%|██████████| 18.7MB / 18.7MB, 1.55MB/s  


Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/emidiosouza/assistente-maternidade-gguf
Unsloth: Cleaning up temporary files...
merged e GGUF publicados
